In [ ]:
CUDA_VISIBLE_DEVICES=0 MAX_PIXELS=1605632 \
swift sft \
  --model Qwen/Qwen3-VL-2B-Instruct \
  --dataset /workspace/ms-swift/dataset/session_20260128_173739/tomato_fork_train.ms_swift.jsonl \
  --val_dataset /workspace/ms-swift/dataset/session_20260128_173739/tomato_fork_val.ms_swift.jsonl \
  --train_type lora \
  --target_modules all-linear \
  --torch_dtype bfloat16 \
  --num_train_epochs 50 \
  --per_device_train_batch_size 4 \
  --gradient_accumulation_steps 4 \
  --learning_rate 2e-4 \
  --lora_rank 16 \
  --lora_alpha 32 \
  --lora_dropout 0.05 \
  --max_length 2048 \
  --save_strategy epoch \
  --eval_strategy epoch \
  --save_total_limit 2 \
  --logging_steps 5 \
  --warmup_ratio 0.05 \
  --output_dir /workspace/ms-swift/output_tomato_fork

# 全参微调
CUDA_VISIBLE_DEVICES=0 MAX_PIXELS=1605632 \
swift sft \
  --model Qwen/Qwen3-VL-2B-Instruct \
  --dataset /workspace/ms-swift/dataset/session_2026023_142020_norm1000/tomato_fork_train.ms_swift_norm1000.jsonl \
  --val_dataset /workspace/ms-swift/dataset/session_2026023_142020_norm1000/tomato_fork_val.ms_swift_norm1000.jsonl \
  --tuner_type full \
  --freeze_llm false \
  --freeze_vit false \
  --freeze_aligner false \
  --torch_dtype float16 \
  --num_train_epochs 50 \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 16 \
  --learning_rate 1e-5 \
  --max_length 2048 \
  --save_strategy epoch \
  --eval_strategy epoch \
  --save_total_limit 2 \
  --logging_steps 5 \
  --warmup_ratio 0.05 \
  --output_dir /workspace/ms-swift/output_tomato_fork_full


# 多卡全参微调
CUDA_VISIBLE_DEVICES=3,4,5,6 NPROC_PER_NODE=4 MAX_PIXELS=1605632 \
swift sft \
  --model Qwen/Qwen3-VL-2B-Instruct \
  --dataset /workspace/ms-swift/dataset/session_2026023_142020_norm1000/tomato_fork_train.ms_swift_norm1000.jsonl \
  --val_dataset /workspace/ms-swift/dataset/session_2026023_142020_norm1000/tomato_fork_val.ms_swift_norm1000.jsonl \
  --tuner_type full \
  --freeze_llm false \
  --freeze_vit false \
  --freeze_aligner false \
  --torch_dtype float16 \
  --num_train_epochs 50 \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 16 \
  --learning_rate 1e-5 \
  --max_length 1024 \
  --save_strategy epoch \
  --eval_strategy epoch \
  --save_total_limit 2 \
  --logging_steps 5 \
  --warmup_ratio 0.05 \
  --output_dir /workspace/ms-swift/output_tomato_fork_full_ddp


In [ ]:
# 推理
CUDA_VISIBLE_DEVICES=0 MAX_PIXELS=1605632 \
swift infer \
  --adapters /workspace/ms-swift/output_tomato_fork/v10-20260203-065051/checkpoint-495 \
  --val_dataset /workspace/ms-swift/dataset/session_2026023_142020/tomato_fork_test.ms_swift.jsonl \
  --result_path /workspace/ms-swift/output_tomato_fork/infer_val.jsonl \
  --max_batch_size 4 \
  --max_new_tokens 256 \
  --temperature 0.0 \
  --stream false

In [ ]:
# 测试
cd /workspace/ms-swift

python /workspace/ms-swift/scripts/render_preds.py \
  --input /workspace/ms-swift/output_tomato_fork/infer_val.jsonl \
  --output_dir /workspace/ms-swift/output_tomato_fork/preds_images \
  --norm_bbox norm1000 

In [ ]:
# 打包推理
# 1. 
swift export \
  --adapters /workspace/ms-swift/output_tomato_fork/v10-20260203-065051/checkpoint-495 \
  --merge_lora true \
  --output_dir /workspace/ms-swift/output_tomato_fork/merged-2_3
# 2.
tar -czvf merged_qwen3_vl_2b.tar.gz /workspace/ms-swift/output_tomato_fork/merged-2_3

# 3.
swift infer \
  --adapters /path/to/merged \
  --query "<image>请标出番茄分叉位置" \
  --images /workspace/ms-swift/dataset/session_20260128_173739/images/fork_369.png

swift infer \
  --model /workspace/ms-swift/output_tomato_fork/merged \
  --query "<image>Highlight the bifurcation points of the tomato stem." \
  --images /workspace/ms-swift/dataset/session_20260128_173739/images/fork_6.png
  

